# Baseline — Patent Phrase Similarity

**Competition:** predict how similar two technical phrases (`anchor`, `target`) are in a
patent context (`context` = patent classification code). Mini version of the Kaggle
*U.S. Patent Phrase to Phrase Matching* competition.

- **Task:** regression, score in [0, 1] (0 = unrelated, 1 = identical meaning)
- **Metric:** Pearson correlation
- **Kaggle link:** _TODO: add link_

**Approach:** TF-IDF character n-grams of both phrases (+ the context code) fed to a
Ridge regressor, plus a hand-made cosine-similarity feature. No deep learning needed
for a solid start.

In [1]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from scipy.stats import pearsonr

DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train.shape, test.shape)
train.head(3)

(22607, 5) (13866, 4)


,id,anchor,target,context,score
0,train_00000,inner peripheral,upper surface,A63,0.25
1,train_00001,storage lid,storage temperature,B44,0.00
2,train_00002,board id,pcb identifier,H05,0.75


In [2]:
# Character n-gram TF-IDF fitted on all phrases, applied to anchor and target separately
all_text = pd.concat([train["anchor"], train["target"], test["anchor"], test["target"]])
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2)
vec.fit(all_text)

def featurize(df):
    A = vec.transform(df["anchor"])
    T = vec.transform(df["target"])
    # cosine similarity between the two phrases (rows are L2-normalized by TF-IDF)
    cos = np.asarray(A.multiply(T).sum(axis=1))
    ctx = pd.get_dummies(df["context"].str[0])  # patent section letter A-H
    return sparse.hstack([A, T, sparse.csr_matrix(cos), sparse.csr_matrix(ctx.values.astype(float))]).tocsr()

# align context dummies between train/test by fitting on the union
X = featurize(train)
Xt = featurize(test)
y = train["score"].values
print(X.shape, Xt.shape)

(22607, 29439) (13866, 29439)


In [3]:
# 5-fold CV with Pearson correlation
oof = np.zeros(len(train))
for tr_idx, va_idx in KFold(5, shuffle=True, random_state=0).split(X):
    model = Ridge(alpha=1.0)
    model.fit(X[tr_idx], y[tr_idx])
    oof[va_idx] = model.predict(X[va_idx])
oof = np.clip(oof, 0, 1)
print(f"CV Pearson: {pearsonr(y, oof)[0]:.4f}")

CV Pearson: 0.5551


In [4]:
model = Ridge(alpha=1.0).fit(X, y)
pred = np.clip(model.predict(Xt), 0, 1)
sub = pd.DataFrame({"id": test["id"], "score": pred})
sub.to_csv("submission.csv", index=False)
sub.head()

,id,score
0,test_00000,0.124089
1,test_00001,0.328429
2,test_00002,0.325133
3,test_00003,0.416358
4,test_00004,0.348554


## Ideas to improve

- **Sentence-transformer embeddings** (e.g. `all-MiniLM-L6-v2` or the patent-specific
  `anferico/bert-for-patents`) — encode anchor/target, use cosine + concat features.
- Fine-tune **DeBERTa-v3** as a cross-encoder on `anchor [SEP] target [SEP] context`
  (this is what won the original Kaggle competition).
- Add word-overlap features (Jaccard, common-token counts) and per-context statistics.
